In [1]:
import pandas as pd
import pickle

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

In [2]:
pre_data = pd.read_csv("preprocessed.csv")

In [3]:
#文字列を含む列のエンコード
categorical_columns = ["Cabin", "Embarked", "Ticket_ID"]
categorical_indices = [pre_data.columns.get_loc(col) for col in categorical_columns]

unique_values_dict = {col: pre_data[col].unique() for col in categorical_columns}

for col in categorical_columns:
    pre_data[col] = pre_data[col].map(lambda x: x if x in unique_values_dict[col] else "other")

column_trans = ColumnTransformer(
    [("onehot", OneHotEncoder(), categorical_indices)], 
    remainder="passthrough"
) 

column_trans.fit(pre_data)
data_encoded = column_trans.transform(pre_data)

In [4]:
#エンコードしたデータの分割
test = data_encoded[891:]

In [5]:
print("test shape:", test.shape)

test shape: (418, 56)


In [6]:
#モデルの読み込み
with open("Titanic_model.pickle", "rb") as fp:
    model = pickle.load(fp)

In [7]:
inference_survived = model.predict(test)

In [8]:
inference_df = pd.read_csv("test.csv")

In [11]:
inference = pd.DataFrame()
inference["PassengerId"] = inference_df["PassengerId"]

In [12]:
inference["Survived"] = inference_survived

In [13]:
inference.to_csv("inference.csv", index=False)